In [ ]:
"""
Enhanced SoundCloud Downloader with On-the-Fly Transcription
Fixed ffmpeg detection and structured data output with comprehensive metadata.
"""

import os
import re
import subprocess
import sys
from datetime import datetime, timedelta
import yt_dlp
import requests
import time
import hashlib
import json
import shutil
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Union
import io
import tempfile
from contextlib import contextmanager
import pandas as pd
import csv

# Audio processing imports
try:
    from pydub import AudioSegment
    from pydub.utils import which
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False
    print("Warning: pydub not available. Install with: pip install pydub")

# Transcription imports
try:
    from google import genai
    from google.genai import types
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print("Warning: Google Gemini not available. Install with: pip install google-generativeai")

try:
    import whisper
    WHISPER_AVAILABLE = True
except ImportError:
    WHISPER_AVAILABLE = False

try:
    from dotenv import load_dotenv
    DOTENV_AVAILABLE = True
except ImportError:
    DOTENV_AVAILABLE = False
    print("Warning: python-dotenv not available. Install with: pip install python-dotenv")


class MemoryAudioProcessor:
    """Handles audio processing in memory without disk storage."""
    
    def __init__(self, ffmpeg_path: Optional[str] = None):
        self.max_memory_size = 100 * 1024 * 1024  # 100MB limit
        self.ffmpeg_path = ffmpeg_path
    
    def set_ffmpeg_path(self, ffmpeg_path: str):
        """Set the ffmpeg path for audio processing."""
        self.ffmpeg_path = ffmpeg_path
        if PYDUB_AVAILABLE:
            # Set ffmpeg path for pydub
            AudioSegment.converter = ffmpeg_path
            AudioSegment.ffmpeg = ffmpeg_path
            AudioSegment.ffprobe = ffmpeg_path.replace('ffmpeg', 'ffprobe')
    
    def normalize_audio_format(self, audio_data: bytes, target_format: str = "mp3") -> bytes:
        """
        Convert audio data to specified format in memory.
        
        Args:
            audio_data (bytes): Raw audio data
            target_format (str): Target format ('mp3', 'wav', 'flac')
            
        Returns:
            bytes: Converted audio data
            
        Raises:
            RuntimeError: If pydub is not available or conversion fails
        """
        if not PYDUB_AVAILABLE:
            raise RuntimeError("pydub is required for audio format conversion")
        
        try:
            # Load audio from bytes
            audio_segment = AudioSegment.from_file(io.BytesIO(audio_data))
            
            # Convert to target format
            output_buffer = io.BytesIO()
            audio_segment.export(output_buffer, format=target_format)
            
            return output_buffer.getvalue()
            
        except Exception as e:
            raise RuntimeError(f"Audio format conversion failed: {e}")
    
    def optimize_audio_for_transcription(self, audio_data: bytes) -> bytes:
        """
        Optimize audio data for better transcription accuracy.
        
        Args:
            audio_data (bytes): Raw audio data
            
        Returns:
            bytes: Optimized audio data
        """
        if not PYDUB_AVAILABLE:
            return audio_data  # Return as-is if pydub not available
        
        try:
            # Load audio
            audio = AudioSegment.from_file(io.BytesIO(audio_data))
            
            # Normalize audio for better transcription
            # Convert to mono if stereo
            if audio.channels > 1:
                audio = audio.set_channels(1)
            
            # Normalize sample rate to 16kHz (good for speech)
            if audio.frame_rate != 16000:
                audio = audio.set_frame_rate(16000)
            
            # Normalize volume
            audio = audio.normalize()
            
            # Export optimized audio
            output_buffer = io.BytesIO()
            audio.export(output_buffer, format="wav")
            
            return output_buffer.getvalue()
            
        except Exception as e:
            print(f"Warning: Audio optimization failed: {e}")
            return audio_data  # Return original if optimization fails


class TranscriptionEngine:
    """Handles different transcription methods."""
    
    def __init__(self, method: str = "gemini"):
        """
        Initialize transcription engine.
        
        Args:
            method (str): Transcription method ('gemini', 'whisper')
        """
        self.method = method
        self.client = None
        
        # Load environment variables first
        if DOTENV_AVAILABLE:
            self._load_environment()
        
        if method == "gemini":
            self._setup_gemini()
        elif method == "whisper":
            self._setup_whisper()
    
    def _load_environment(self):
        """Load environment variables from .env file."""
        # Try different possible locations for .env file
        possible_env_paths = [
            ".env",  # Current directory
            "../.env",  # One level up (for notebooks)
            "../../.env",  # Two levels up
            "/teamspace/studios/this_studio/.env",  # Your specific path
            "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/.env",  # Project root
        ]
        
        env_loaded = False
        for env_path in possible_env_paths:
            if os.path.exists(env_path):
                try:
                    load_dotenv(dotenv_path=env_path)
                    api_key = os.getenv("GEMINI_API_KEY")
                    if api_key:
                        print(f"✓ Loaded API key from: {env_path}")
                        env_loaded = True
                        break
                except Exception as e:
                    print(f"Warning: Error loading {env_path}: {e}")
        
        if not env_loaded:
            print("Warning: No .env file found with valid GEMINI_API_KEY")
    
    def _setup_gemini(self):
        """Setup Gemini API client."""
        if not GEMINI_AVAILABLE:
            raise RuntimeError("Gemini API not available. Install google-generativeai")
        
        api_key = os.getenv("GEMINI_API_KEY")
        if not api_key:
            print("\nERROR: GEMINI_API_KEY not found!")
            print("Quick fix:")
            print("1. Get API key from: https://aistudio.google.com")
            print("2. Create .env file in project root:")
            print('   echo \'GEMINI_API_KEY="your_key_here"\' > .env')
            print("3. Or set environment variable:")
            print('   export GEMINI_API_KEY="your_key_here"')
            raise EnvironmentError("GEMINI_API_KEY not found in environment variables")
        
        self.client = genai.Client(api_key=api_key)
    
    def _setup_whisper(self):
        """Setup Whisper model."""
        if not WHISPER_AVAILABLE:
            raise RuntimeError("Whisper not available. Install with: pip install openai-whisper")
        
        try:
            self.client = whisper.load_model("base")
        except Exception as e:
            raise RuntimeError(f"Failed to load Whisper model: {e}")
    
    def transcribe_from_memory(
        self, 
        audio_data: bytes, 
        model: str = "gemini-2.0-flash"
    ) -> str:
        """
        Transcribe audio data from memory.
        
        Args:
            audio_data (bytes): Audio data in memory
            model (str): Model to use for transcription
            
        Returns:
            str: Transcribed text
            
        Raises:
            ValueError: If transcription fails or returns empty result
        """
        if self.method == "gemini":
            return self._transcribe_gemini(audio_data, model)
        elif self.method == "whisper":
            return self._transcribe_whisper(audio_data)
        else:
            raise ValueError(f"Unsupported transcription method: {self.method}")
    
    def _transcribe_gemini(self, audio_data: bytes, model: str) -> str:
        """Transcribe using Gemini API."""
        try:
            # Create audio part from bytes
            audio_part = types.Part.from_bytes(
                data=audio_data,
                mime_type="audio/mp3"
            )
            
            # Generate transcript
            response = self.client.models.generate_content(
                model=model,
                contents=["Generate a transcript of the speech.", audio_part]
            )
            
            if hasattr(response, 'text') and response.text:
                return response.text
            else:
                raise ValueError("No transcript text returned from Gemini API")
                
        except Exception as e:
            raise ValueError(f"Gemini transcription failed: {e}")
    
    def _transcribe_whisper(self, audio_data: bytes) -> str:
        """Transcribe using Whisper."""
        try:
            # Whisper requires a file, so we use a temporary file
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_file:
                tmp_file.write(audio_data)
                tmp_file_path = tmp_file.name
            
            try:
                result = self.client.transcribe(tmp_file_path)
                return result["text"]
            finally:
                # Clean up temporary file
                os.unlink(tmp_file_path)
                
        except Exception as e:
            raise ValueError(f"Whisper transcription failed: {e}")


class StreamingSoundCloudDownloader:
    """Enhanced SoundCloud downloader with structured data output and fixed ffmpeg detection."""
    
    def __init__(
        self, 
        output_dir: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw",
        transcription_method: str = "gemini",
        output_format: str = "structured"  # 'structured', 'txt', or 'both'
    ):
        """
        Initialize the streaming downloader with enhanced ffmpeg detection.
        
        Args:
            output_dir (str): Directory for output files
            transcription_method (str): Method for transcription ('gemini', 'whisper')
            output_format (str): Output format ('structured', 'txt', 'both')
        """
        self.output_dir = output_dir
        self.transcription_method = transcription_method
        self.output_format = output_format
        self.ffmpeg_path = None
        
        # Create directory structure
        os.makedirs(output_dir, exist_ok=True)
        
        # Initialize log files
        self.transcription_log = {}
        self.log_file = os.path.join(output_dir, ".transcription_log.json")
        self.structured_data_file = os.path.join(output_dir, "transcriptions_database.csv")
        
        # Load transcription log
        self.load_transcription_log()
        
        # Setup ffmpeg with improved detection
        self.setup_ffmpeg()
        
        # Initialize components
        self.audio_processor = MemoryAudioProcessor(self.ffmpeg_path)
        self.transcription_engine = TranscriptionEngine(transcription_method)
        
        # Initialize structured data storage
        self.init_structured_storage()
    
    def find_ffmpeg(self):
        """Find ffmpeg binary in system - improved detection."""
        # Check if ffmpeg is in PATH using shutil.which
        ffmpeg_in_path = shutil.which('ffmpeg')
        if ffmpeg_in_path:
            return ffmpeg_in_path
        
        # Common locations to check
        common_locations = [
            '/usr/bin/ffmpeg',
            '/usr/local/bin/ffmpeg',
            '/opt/homebrew/bin/ffmpeg',
            '/snap/bin/ffmpeg',
            '~/.local/bin/ffmpeg',
            '/conda/bin/ffmpeg',
            '/miniconda/bin/ffmpeg',
            '/home/zeus/miniconda3/bin/ffmpeg',
            '/home/zeus/miniconda3/envs/cloudspace/bin/ffmpeg',
        ]
        
        for location in common_locations:
            expanded_path = os.path.expanduser(location)
            if os.path.exists(expanded_path) and os.access(expanded_path, os.X_OK):
                return expanded_path
        
        return None
    
    def verify_ffmpeg(self, ffmpeg_path):
        """Verify that ffmpeg actually works."""
        try:
            result = subprocess.run(
                [ffmpeg_path, '-version'],
                capture_output=True,
                text=True,
                timeout=5
            )
            return result.returncode == 0 and 'ffmpeg version' in result.stdout
        except Exception as e:
            print(f"Error verifying ffmpeg: {e}")
            return False
    
    def install_ffmpeg(self):
        """Try to install ffmpeg based on detected OS."""
        try:
            # Check if we're in a conda environment
            if 'CONDA_DEFAULT_ENV' in os.environ or 'conda' in sys.executable.lower():
                print("Attempting to install ffmpeg using conda...")
                result = subprocess.run(['conda', 'install', '-y', '-c', 'conda-forge', 'ffmpeg'], 
                                      capture_output=True, text=True)
                if result.returncode == 0:
                    return True
            
            # Detect the Linux distribution
            if os.path.exists('/etc/os-release'):
                with open('/etc/os-release', 'r') as f:
                    os_info = f.read().lower()
                
                if 'ubuntu' in os_info or 'debian' in os_info:
                    print("Installing ffmpeg using apt...")
                    subprocess.run(['sudo', 'apt-get', 'update'], check=False, capture_output=True)
                    result = subprocess.run(['sudo', 'apt-get', 'install', '-y', 'ffmpeg'], 
                                          capture_output=True, text=True)
                    return result.returncode == 0
                    
                elif 'fedora' in os_info:
                    print("Installing ffmpeg using dnf...")
                    result = subprocess.run(['sudo', 'dnf', 'install', '-y', 'ffmpeg'], 
                                          capture_output=True, text=True)
                    return result.returncode == 0
                    
                elif 'arch' in os_info:
                    print("Installing ffmpeg using pacman...")
                    result = subprocess.run(['sudo', 'pacman', '-S', '--noconfirm', 'ffmpeg'], 
                                          capture_output=True, text=True)
                    return result.returncode == 0
        except Exception as e:
            print(f"Error during installation: {e}")
        
        return False
    
    def setup_ffmpeg(self):
        """Setup ffmpeg for Linux systems with improved detection."""
        print("🔧 Checking for ffmpeg installation...")
        
        # First try to find existing ffmpeg
        self.ffmpeg_path = self.find_ffmpeg()
        
        if self.ffmpeg_path:
            # Verify it actually works
            if self.verify_ffmpeg(self.ffmpeg_path):
                print(f"✓ Found working ffmpeg at: {self.ffmpeg_path}")
                return True
            else:
                print(f"⚠ Found ffmpeg at {self.ffmpeg_path} but it doesn't work properly")
                self.ffmpeg_path = None
        
        # If not found, try to install it
        print("📦 ffmpeg not found. Attempting to install...")
        
        if self.install_ffmpeg():
            # Re-check after installation
            self.ffmpeg_path = self.find_ffmpeg()
            if self.ffmpeg_path and self.verify_ffmpeg(self.ffmpeg_path):
                print(f"✓ Successfully installed ffmpeg at: {self.ffmpeg_path}")
                return True
        
        print("⚠ Failed to install ffmpeg automatically.")
        print("💡 Please install it manually:")
        print("  • Conda: conda install -c conda-forge ffmpeg")
        print("  • Ubuntu/Debian: sudo apt-get install ffmpeg")
        print("  • Fedora: sudo dnf install ffmpeg")
        print("  • Arch: sudo pacman -S ffmpeg")
        return False
    
    def init_structured_storage(self):
        """Initialize structured data storage (CSV database)."""
        if not os.path.exists(self.structured_data_file):
            # Create CSV with comprehensive columns
            columns = [
                'id', 'url', 'title', 'source_type', 'date_recorded', 'date_processed',
                'transcription_method', 'model_used', 'processing_duration_seconds',
                'audio_size_mb', 'audio_duration_seconds', 'audio_format', 'audio_sample_rate',
                'transcript_length_chars', 'transcript_length_words', 'language_detected',
                'confidence_score', 'processing_status', 'error_message', 'file_path',
                'transcript_text'
            ]
            
            df = pd.DataFrame(columns=columns)
            df.to_csv(self.structured_data_file, index=False)
            print(f"✓ Created structured database: {self.structured_data_file}")
    
    def load_transcription_log(self):
        """Load the transcription log from file."""
        if os.path.exists(self.log_file):
            try:
                with open(self.log_file, 'r') as f:
                    self.transcription_log = json.load(f)
            except:
                self.transcription_log = {}
    
    def save_transcription_log(self):
        """Save the transcription log to file."""
        try:
            with open(self.log_file, 'w') as f:
                json.dump(self.transcription_log, f, indent=2)
        except Exception as e:
            print(f"Warning: Could not save transcription log: {e}")
    
    @contextmanager
    def memory_buffer_manager(self):
        """Context manager for handling memory buffers safely."""
        buffer = io.BytesIO()
        try:
            yield buffer
        finally:
            buffer.close()
    
    def download_to_memory(self, url: str) -> Tuple[Optional[bytes], Dict]:
        """
        Download audio from URL directly to memory buffer with metadata.
        
        Args:
            url (str): SoundCloud URL to download
            
        Returns:
            Tuple[Optional[bytes], Dict]: Audio data and metadata
        """
        print(f"⬇ Downloading to memory: {url}")
        metadata = {'success': False, 'error': None, 'info': {}}
        
        # Configure yt-dlp for memory download
        ydl_opts = {
            'format': 'bestaudio/best',
            'noplaylist': True,
            'quiet': True,
            'no_warnings': True,
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192',
            }],
        }
        
        if self.ffmpeg_path:
            ffmpeg_dir = os.path.dirname(self.ffmpeg_path)
            ydl_opts['ffmpeg_location'] = ffmpeg_dir
        
        try:
            # Get info first
            with yt_dlp.YoutubeDL({'quiet': True}) as ydl:
                info = ydl.extract_info(url, download=False)
                metadata['info'] = {
                    'title': info.get('title', 'Unknown'),
                    'duration': info.get('duration', 0),
                    'uploader': info.get('uploader', 'Unknown'),
                    'upload_date': info.get('upload_date', ''),
                    'view_count': info.get('view_count', 0),
                    'like_count': info.get('like_count', 0),
                }
            
            # Download to temporary file
            with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as tmp_file:
                ydl_opts['outtmpl'] = tmp_file.name.replace('.mp3', '.%(ext)s')
                
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([url])
                
                # Read the processed file into memory
                processed_file = tmp_file.name.replace('.mp3', '.mp3')
                if os.path.exists(processed_file):
                    with open(processed_file, 'rb') as f:
                        audio_data = f.read()
                    
                    # Clean up temporary file immediately
                    os.unlink(processed_file)
                    metadata['success'] = True
                    return audio_data, metadata
                    
        except Exception as e:
            metadata['error'] = str(e)
            print(f"  ✗ Download failed: {str(e)[:100]}")
            return None, metadata
        
        return None, metadata
    
    def transcribe_audio_data(
        self, 
        audio_data: bytes, 
        metadata: Dict,
        optimize: bool = True
    ) -> Tuple[Optional[str], Dict]:
        """
        Transcribe audio data from memory buffer with comprehensive metadata.
        
        Args:
            audio_data (bytes): Raw audio data
            metadata (Dict): Existing metadata
            optimize (bool): Whether to optimize audio for transcription
            
        Returns:
            Tuple[Optional[str], Dict]: Transcribed text and updated metadata
        """
        start_time = time.time()
        
        if not audio_data or len(audio_data) < 1000:
            metadata['transcription_error'] = "Audio data is too small or empty"
            return None, metadata
        
        try:
            # Get audio metadata
            if PYDUB_AVAILABLE:
                try:
                    audio = AudioSegment.from_file(io.BytesIO(audio_data))
                    metadata['audio_duration_seconds'] = len(audio) / 1000.0
                    metadata['audio_sample_rate'] = audio.frame_rate
                    metadata['audio_channels'] = audio.channels
                    metadata['audio_format'] = 'mp3'
                except:
                    pass
            
            metadata['audio_size_mb'] = len(audio_data) / (1024 * 1024)
            
            # Optimize audio if requested
            if optimize and PYDUB_AVAILABLE and self.ffmpeg_path:
                print("  🔧 Optimizing audio for transcription...")
                audio_data = self.audio_processor.optimize_audio_for_transcription(audio_data)
            
            # Check memory usage
            memory_mb = len(audio_data) / (1024 * 1024)
            print(f"  📊 Processing {memory_mb:.1f}MB audio in memory")
            
            if memory_mb > 100:
                print("  ⚠ Large audio file, processing may take longer")
            
            # Transcribe
            print("  🎯 Generating transcript...")
            transcript = self.transcription_engine.transcribe_from_memory(audio_data)
            
            # Calculate processing time and metadata
            processing_time = time.time() - start_time
            metadata.update({
                'processing_duration_seconds': processing_time,
                'transcript_length_chars': len(transcript) if transcript else 0,
                'transcript_length_words': len(transcript.split()) if transcript else 0,
                'transcription_success': True,
                'transcription_method': self.transcription_method,
            })
            
            return transcript, metadata
            
        except Exception as e:
            metadata.update({
                'transcription_error': str(e),
                'transcription_success': False,
                'processing_duration_seconds': time.time() - start_time,
            })
            print(f"  ✗ Transcription error: {e}")
            return None, metadata
    
    def save_structured_data(self, record: Dict):
        """Save transcription data to structured CSV database."""
        try:
            # Read existing data
            if os.path.exists(self.structured_data_file):
                df = pd.read_csv(self.structured_data_file)
            else:
                df = pd.DataFrame()
            
            # Create new record
            new_row = pd.DataFrame([record])
            
            # Append to existing data
            df = pd.concat([df, new_row], ignore_index=True)
            
            # Save back to CSV
            df.to_csv(self.structured_data_file, index=False)
            
        except Exception as e:
            print(f"Warning: Could not save to structured database: {e}")
    
    def save_transcript_file(self, transcript: str, metadata: Dict, base_filename: str) -> str:
        """Save transcript as text file with metadata header."""
        transcript_file = os.path.join(self.output_dir, f"{base_filename}.txt")
        
        # Handle filename conflicts
        counter = 1
        original_path = transcript_file
        while os.path.exists(transcript_file):
            name_part = original_path.replace('.txt', '')
            transcript_file = f"{name_part}_{counter}.txt"
            counter += 1
        
        # Save transcript with comprehensive metadata
        with open(transcript_file, 'w', encoding='utf-8') as f:
            f.write(f"# Audio Transcription Report\n")
            f.write(f"{'='*50}\n\n")
            f.write(f"**Source URL**: {metadata.get('url', 'Unknown')}\n")
            f.write(f"**Title**: {metadata.get('info', {}).get('title', 'Unknown')}\n")
            f.write(f"**Date Processed**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"**Transcription Method**: {self.transcription_method}\n")
            f.write(f"**Audio Duration**: {metadata.get('audio_duration_seconds', 0):.1f} seconds\n")
            f.write(f"**Audio Size**: {metadata.get('audio_size_mb', 0):.1f} MB\n")
            f.write(f"**Processing Time**: {metadata.get('processing_duration_seconds', 0):.1f} seconds\n")
            f.write(f"**Transcript Length**: {metadata.get('transcript_length_words', 0)} words\n\n")
            f.write(f"{'='*50}\n\n")
            f.write("## Transcript\n\n")
            f.write(transcript)
        
        return transcript_file
    
    def process_url_streaming(
        self, 
        url: str, 
        custom_filename: Optional[str] = None
    ) -> Tuple[bool, Optional[str], Dict]:
        """
        Complete streaming workflow with structured data output.
        
        Args:
            url (str): SoundCloud URL to process
            custom_filename (str, optional): Custom name for output file
            
        Returns:
            Tuple[bool, Optional[str], Dict]: (success, file_path, metadata)
        """
        # Generate URL hash for tracking
        url_hash = hashlib.md5(url.encode()).hexdigest()
        
        # Check if already processed
        if url_hash in self.transcription_log:
            existing_record = self.transcription_log[url_hash]
            if self.output_format == 'structured':
                print(f"✓ Already processed: {existing_record.get('title', 'Unknown')}")
                return True, existing_record.get('structured_record_id'), existing_record
            else:
                existing_file = existing_record.get('transcript_path')
                if existing_file and os.path.exists(existing_file):
                    print(f"✓ Already transcribed: {os.path.basename(existing_file)}")
                    return True, existing_file, existing_record
        
        print(f"🔄 Processing: {url}")
        
        # Step 1: Download to memory
        audio_data, metadata = self.download_to_memory(url)
        if not audio_data:
            return False, None, metadata
        
        metadata['url'] = url
        metadata['id'] = url_hash
        
        try:
            # Step 2: Transcribe from memory
            transcript, metadata = self.transcribe_audio_data(audio_data, metadata)
            if not transcript:
                return False, None, metadata
            
            # Step 3: Generate filename
            if custom_filename:
                base_filename = custom_filename
            else:
                base_filename = self.extract_safe_filename(metadata.get('info', {}).get('title', ''), url)
            
            # Step 4: Save based on output format
            file_paths = []
            
            if self.output_format in ['structured', 'both']:
                # Save to structured database
                structured_record = {
                    'id': url_hash,
                    'url': url,
                    'title': metadata.get('info', {}).get('title', 'Unknown'),
                    'source_type': 'soundcloud',
                    'date_recorded': metadata.get('info', {}).get('upload_date', ''),
                    'date_processed': datetime.now().isoformat(),
                    'transcription_method': self.transcription_method,
                    'model_used': 'gemini-2.0-flash' if self.transcription_method == 'gemini' else 'whisper-base',
                    'processing_duration_seconds': metadata.get('processing_duration_seconds', 0),
                    'audio_size_mb': metadata.get('audio_size_mb', 0),
                    'audio_duration_seconds': metadata.get('audio_duration_seconds', 0),
                    'audio_format': metadata.get('audio_format', 'mp3'),
                    'audio_sample_rate': metadata.get('audio_sample_rate', 0),
                    'transcript_length_chars': metadata.get('transcript_length_chars', 0),
                    'transcript_length_words': metadata.get('transcript_length_words', 0),
                    'language_detected': 'somali',  # Assuming based on your project
                    'confidence_score': '',  # Could be added later
                    'processing_status': 'success',
                    'error_message': '',
                    'file_path': f"{base_filename}.txt" if self.output_format == 'both' else '',
                    'transcript_text': transcript
                }
                
                self.save_structured_data(structured_record)
                file_paths.append(self.structured_data_file)
                print(f"  ✓ Added to structured database: {base_filename}")
            
            if self.output_format in ['txt', 'both']:
                # Save as text file
                transcript_file = self.save_transcript_file(transcript, metadata, base_filename)
                file_paths.append(transcript_file)
                print(f"  ✓ Transcript saved: {os.path.basename(transcript_file)}")
            
            # Update log
            self.transcription_log[url_hash] = {
                'url': url,
                'title': metadata.get('info', {}).get('title', 'Unknown'),
                'transcript_path': file_paths[0] if file_paths else None,
                'structured_record_id': url_hash,
                'processing_date': datetime.now().isoformat(),
                'method': self.transcription_method,
                'file_size_mb': metadata.get('audio_size_mb', 0),
                'success': True
            }
            self.save_transcription_log()
            
            return True, file_paths[0] if file_paths else None, metadata
            
        finally:
            # Memory cleanup
            del audio_data  # Explicit cleanup
    
    def extract_safe_filename(self, title: str, url: str) -> str:
        """Extract a safe filename from title or URL."""
        if title and title != 'Unknown':
            # Clean the title
            safe_filename = re.sub(r'[^\w\s-]', '', title)
            safe_filename = re.sub(r'[-\s]+', '-', safe_filename).strip('-')
            if safe_filename:
                return safe_filename
        
        # Fallback: extract from URL
        parts = url.split('/')
        if len(parts) >= 2:
            return f"{parts[-2]}_{parts[-1]}"
        
        # Final fallback
        return f"soundcloud_audio_{int(time.time())}"
    
    def process_date_range_streaming(
        self,
        profile_url: str,
        start_date: Union[str, datetime],
        end_date: Union[str, datetime],
        batch_size: int = 5
    ) -> Dict[str, List]:
        """
        Process date range with streaming transcription and structured output.
        
        Args:
            profile_url (str): SoundCloud profile URL
            start_date: Start date for range
            end_date: End date for range
            batch_size (int): Number of files to process before memory cleanup
            
        Returns:
            Dict[str, List]: Results categorized by success/failure
        """
        # Parse dates if strings
        if isinstance(start_date, str):
            start_date = datetime.strptime(start_date, '%Y-%m-%d')
        if isinstance(end_date, str):
            end_date = datetime.strptime(end_date, '%Y-%m-%d')
        
        print(f"\n🎵 Streaming transcription: {start_date.date()} to {end_date.date()}")
        print(f"📁 Output directory: {self.output_dir}")
        print(f"🧠 Using {self.transcription_method} for transcription")
        print(f"📊 Output format: {self.output_format}")
        if self.ffmpeg_path:
            print(f"🔧 Using ffmpeg: {self.ffmpeg_path}")
        print()
        
        # Generate URLs
        urls = self.generate_urls_for_range(profile_url, start_date, end_date)
        
        if not urls:
            print("No URLs generated for date range")
            return {'successful': [], 'failed': [], 'skipped': []}
        
        results = {'successful': [], 'failed': [], 'skipped': [], 'metadata': []}
        batch_count = 0
        
        for i, (date, url) in enumerate(urls, 1):
            print(f"[{i}/{len(urls)}] {date.date()}")
            
            try:
                success, file_path, metadata = self.process_url_streaming(url)
                
                if success and file_path:
                    results['successful'].append(file_path)
                    results['metadata'].append(metadata)
                else:
                    results['failed'].append({'url': url, 'date': date, 'error': metadata.get('error', 'Unknown')})
                    
            except Exception as e:
                print(f"  ✗ Processing error: {e}")
                results['failed'].append({'url': url, 'date': date, 'error': str(e)})
            
            # Memory management: cleanup every batch_size items
            batch_count += 1
            if batch_count >= batch_size:
                print(f"  🧹 Memory cleanup after {batch_size} items...")
                import gc
                gc.collect()
                batch_count = 0
            
            # Rate limiting
            time.sleep(2)
        
        # Print final summary
        self.print_processing_summary(results, len(urls))
        return results
    
    def generate_urls_for_range(
        self, 
        profile_url: str, 
        start_date: datetime, 
        end_date: datetime
    ) -> List[Tuple[datetime, str]]:
        """Generate potential URLs for a date range."""
        urls = []
        profile_url = profile_url.rstrip('/')
        
        current_date = start_date
        while current_date <= end_date:
            day = current_date.day
            month = current_date.strftime('%b').lower()
            year = current_date.year
            
            # Generate URL (using your existing pattern)
            url = f"{profile_url}/idaacadda-{day:02d}-{month}-{year}"
            urls.append((current_date, url))
            
            current_date += timedelta(days=1)
        
        return urls
    
    def print_processing_summary(self, results: Dict[str, List], total_urls: int):
        """Print a comprehensive summary of processing results."""
        successful = len(results['successful'])
        failed = len(results['failed'])
        skipped = len(results['skipped'])
        
        print(f"\n{'='*70}")
        print(f"🎵 STREAMING TRANSCRIPTION SUMMARY")
        print(f"{'='*70}")
        print(f"  📊 Total URLs processed: {total_urls}")
        print(f"  ✅ Successfully transcribed: {successful}")
        print(f"  ⏭️  Skipped (already done): {skipped}")
        print(f"  ❌ Failed: {failed}")
        print(f"  📁 Output directory: {self.output_dir}")
        print(f"  📊 Output format: {self.output_format}")
        
        if self.output_format in ['structured', 'both']:
            print(f"  🗃️  Structured database: {os.path.basename(self.structured_data_file)}")
        
        if results['metadata']:
            # Calculate some statistics
            total_duration = sum(m.get('audio_duration_seconds', 0) for m in results['metadata'])
            total_words = sum(m.get('transcript_length_words', 0) for m in results['metadata'])
            total_processing_time = sum(m.get('processing_duration_seconds', 0) for m in results['metadata'])
            
            print(f"  🎧 Total audio duration: {total_duration/60:.1f} minutes")
            print(f"  📝 Total words transcribed: {total_words:,}")
            print(f"  ⏱️  Total processing time: {total_processing_time/60:.1f} minutes")
        
        print(f"  💾 Memory-efficient processing: ✓")
        print(f"  🔧 ffmpeg status: {'✓' if self.ffmpeg_path else '⚠ Not found'}")
        print(f"{'='*70}\n")
        
        # Print failed URLs if any
        if results['failed']:
            print("❌ Failed URLs:")
            for failed in results['failed'][:5]:  # Show first 5 failures
                if isinstance(failed, dict):
                    print(f"  • {failed.get('date', 'Unknown date')}: {failed.get('error', 'Unknown error')[:50]}...")
                else:
                    print(f"  • {failed}")
            if len(results['failed']) > 5:
                print(f"  ... and {len(results['failed']) - 5} more")
            print()
    
    def validate_url(self, url: str) -> bool:
        """Validate SoundCloud URL format."""
        pattern = r'^https?://(?:www\.)?soundcloud\.com/[\w-]+/[\w-]+'
        return bool(re.match(pattern, url))
    
    def batch_transcribe_existing_files(
        self, 
        audio_dir: str, 
        batch_size: int = 3
    ) -> Dict[str, str]:
        """
        Transcribe existing MP3 files with structured output.
        
        Args:
            audio_dir (str): Directory containing MP3 files
            batch_size (int): Number of files to process before memory cleanup
            
        Returns:
            Dict[str, str]: Processing results per file
        """
        mp3_files = list(Path(audio_dir).glob("*.mp3"))
        results = {}
        
        print(f"📁 Found {len(mp3_files)} MP3 files to transcribe")
        print(f"📝 Batch size: {batch_size} files")
        print(f"📊 Output format: {self.output_format}")
        
        for i, mp3_file in enumerate(mp3_files, 1):
            print(f"\n[{i}/{len(mp3_files)}] Processing {mp3_file.name}")
            
            try:
                # Read file to memory
                with open(mp3_file, 'rb') as f:
                    audio_data = f.read()
                
                # Create metadata
                metadata = {
                    'url': f"file://{mp3_file.absolute()}",
                    'id': hashlib.md5(str(mp3_file).encode()).hexdigest(),
                    'info': {'title': mp3_file.stem}
                }
                
                # Transcribe
                transcript, metadata = self.transcribe_audio_data(audio_data, metadata)
                
                if transcript:
                    # Save based on output format
                    if self.output_format in ['structured', 'both']:
                        structured_record = {
                            'id': metadata['id'],
                            'url': metadata['url'],
                            'title': mp3_file.stem,
                            'source_type': 'local_file',
                            'date_recorded': '',
                            'date_processed': datetime.now().isoformat(),
                            'transcription_method': self.transcription_method,
                            'model_used': 'gemini-2.0-flash' if self.transcription_method == 'gemini' else 'whisper-base',
                            'processing_duration_seconds': metadata.get('processing_duration_seconds', 0),
                            'audio_size_mb': metadata.get('audio_size_mb', 0),
                            'audio_duration_seconds': metadata.get('audio_duration_seconds', 0),
                            'audio_format': 'mp3',
                            'audio_sample_rate': metadata.get('audio_sample_rate', 0),
                            'transcript_length_chars': metadata.get('transcript_length_chars', 0),
                            'transcript_length_words': metadata.get('transcript_length_words', 0),
                            'language_detected': 'somali',
                            'confidence_score': '',
                            'processing_status': 'success',
                            'error_message': '',
                            'file_path': f"{mp3_file.stem}.txt" if self.output_format == 'both' else '',
                            'transcript_text': transcript
                        }
                        
                        self.save_structured_data(structured_record)
                        print(f"  ✓ Added to structured database")
                    
                    if self.output_format in ['txt', 'both']:
                        output_file = os.path.join(self.output_dir, f"{mp3_file.stem}.txt")
                        
                        with open(output_file, 'w', encoding='utf-8') as f:
                            f.write(f"# Transcript for {mp3_file.name}\n")
                            f.write(f"**Source**: {mp3_file.absolute()}\n")
                            f.write(f"**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                            f.write(f"**Method**: {self.transcription_method}\n\n")
                            f.write(transcript)
                        
                        print(f"  ✓ Saved transcript: {os.path.basename(output_file)}")
                    
                    results[mp3_file.name] = "success"
                else:
                    results[mp3_file.name] = "transcription_failed"
                
                # Memory cleanup
                del audio_data
                
                # Batch cleanup
                if i % batch_size == 0:
                    print(f"  🧹 Memory cleanup after {batch_size} files...")
                    import gc
                    gc.collect()
                    time.sleep(1)
                    
            except Exception as e:
                results[mp3_file.name] = f"error: {e}"
                print(f"  ✗ Error: {e}")
        
        print(f"\n✅ Batch processing completed!")
        print(f"📊 Results: {results}")
        return results
    
    def get_transcription_statistics(self) -> Dict:
        """Get comprehensive statistics from the transcription database."""
        try:
            if not os.path.exists(self.structured_data_file):
                return {"error": "No structured database found"}
            
            df = pd.read_csv(self.structured_data_file)
            
            if len(df) == 0:
                return {"message": "No transcriptions in database"}
            
            stats = {
                "total_transcriptions": len(df),
                "successful_transcriptions": len(df[df['processing_status'] == 'success']),
                "failed_transcriptions": len(df[df['processing_status'] != 'success']),
                "total_audio_duration_hours": df['audio_duration_seconds'].sum() / 3600,
                "total_words_transcribed": df['transcript_length_words'].sum(),
                "total_processing_time_hours": df['processing_duration_seconds'].sum() / 3600,
                "average_audio_size_mb": df['audio_size_mb'].mean(),
                "methods_used": df['transcription_method'].value_counts().to_dict(),
                "date_range": {
                    "earliest": df['date_processed'].min(),
                    "latest": df['date_processed'].max()
                },
                "source_types": df['source_type'].value_counts().to_dict()
            }
            
            return stats
            
        except Exception as e:
            return {"error": f"Error calculating statistics: {e}"}


# Enhanced utility functions
def stream_transcribe_date_range(
    profile_url: str,
    start_date: str,
    end_date: str,
    output_dir: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw",
    method: str = "gemini",
    output_format: str = "structured"
) -> Dict[str, List]:
    """
    Main function for streaming transcription of date ranges with structured output.
    
    Args:
        profile_url (str): SoundCloud profile URL
        start_date (str): Start date in 'YYYY-MM-DD' format
        end_date (str): End date in 'YYYY-MM-DD' format
        output_dir (str): Directory for output files
        method (str): Transcription method ('gemini' or 'whisper')
        output_format (str): Output format ('structured', 'txt', or 'both')
        
    Returns:
        Dict[str, List]: Processing results with metadata
        
    Usage example:
        results = stream_transcribe_date_range(
            profile_url="https://soundcloud.com/radio-ergo",
            start_date="2024-07-01",
            end_date="2024-07-03",
            output_format="structured"
        )
    """
    downloader = StreamingSoundCloudDownloader(
        output_dir=output_dir,
        transcription_method=method,
        output_format=output_format
    )
    
    return downloader.process_date_range_streaming(
        profile_url=profile_url,
        start_date=start_date,
        end_date=end_date
    )

def stream_transcribe_single_url(
    url: str, 
    output_dir: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw",
    method: str = "gemini",
    output_format: str = "structured"
) -> Tuple[bool, Optional[str], Dict]:
    """
    Transcribe a single URL with streaming processing and structured output.
    
    Args:
        url (str): SoundCloud URL to transcribe
        output_dir (str): Output directory
        method (str): Transcription method to use
        output_format (str): Output format ('structured', 'txt', or 'both')
        
    Returns:
        Tuple[bool, Optional[str], Dict]: (success, file_path, metadata)
        
    Usage example:
        success, path, meta = stream_transcribe_single_url(
            "https://soundcloud.com/radio-ergo/idaacadda-01-jul-2024",
            output_format="both"
        )
    """
    downloader = StreamingSoundCloudDownloader(
        output_dir=output_dir,
        transcription_method=method,
        output_format=output_format
    )
    
    return downloader.process_url_streaming(url)

def batch_transcribe_existing_mp3s(
    input_dir: str,
    output_dir: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw",
    method: str = "gemini",
    output_format: str = "structured",
    batch_size: int = 3
) -> Dict[str, str]:
    """
    Batch transcribe existing MP3 files with structured output.
    
    Args:
        input_dir (str): Directory containing MP3 files
        output_dir (str): Directory for output
        method (str): Transcription method to use
        output_format (str): Output format ('structured', 'txt', or 'both')
        batch_size (int): Files to process before memory cleanup
        
    Returns:
        Dict[str, str]: Results per filename
        
    Usage example:
        results = batch_transcribe_existing_mp3s(
            input_dir="./existing_audio",
            output_format="structured"
        )
    """
    downloader = StreamingSoundCloudDownloader(
        output_dir=output_dir,
        transcription_method=method,
        output_format=output_format
    )
    
    return downloader.batch_transcribe_existing_files(input_dir, batch_size)

def get_transcription_database_stats(
    database_path: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/transcriptions_database.csv"
) -> Dict:
    """
    Get comprehensive statistics from the transcription database.
    
    Args:
        database_path (str): Path to the CSV database
        
    Returns:
        Dict: Comprehensive statistics
        
    Usage example:
        stats = get_transcription_database_stats()
        print(f"Total transcriptions: {stats['total_transcriptions']}")
    """
    try:
        if not os.path.exists(database_path):
            return {"error": "Database not found"}
        
        df = pd.read_csv(database_path)
        
        if len(df) == 0:
            return {"message": "No transcriptions in database"}
        
        stats = {
            "total_transcriptions": len(df),
            "successful_transcriptions": len(df[df['processing_status'] == 'success']),
            "total_audio_hours": df['audio_duration_seconds'].sum() / 3600,
            "total_words": df['transcript_length_words'].sum(),
            "methods_used": df['transcription_method'].value_counts().to_dict(),
            "average_processing_time": df['processing_duration_seconds'].mean(),
            "date_range": {
                "earliest": df['date_processed'].min(),
                "latest": df['date_processed'].max()
            }
        }
        
        return stats
        
    except Exception as e:
        return {"error": f"Error reading database: {e}"}

def view_recent_transcriptions(
    database_path: str = "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/transcriptions_database.csv",
    n: int = 10
) -> pd.DataFrame:
    """
    View recent transcriptions from the database.
    
    Args:
        database_path (str): Path to the CSV database
        n (int): Number of recent entries to show
        
    Returns:
        pd.DataFrame: Recent transcriptions
        
    Usage example:
        recent = view_recent_transcriptions(n=5)
        print(recent[['title', 'date_processed', 'transcript_length_words']])
    """
    try:
        df = pd.read_csv(database_path)
        df['date_processed'] = pd.to_datetime(df['date_processed'])
        recent = df.sort_values('date_processed', ascending=False).head(n)
        return recent[['title', 'date_processed', 'transcription_method', 
                      'audio_duration_seconds', 'transcript_length_words', 'processing_status']]
    except Exception as e:
        print(f"Error reading database: {e}")
        return pd.DataFrame()

# Memory and system utilities
def get_memory_usage() -> float:
    """Get current memory usage in MB."""
    try:
        import psutil
        process = psutil.Process()
        memory_mb = process.memory_info().rss / 1024 / 1024
        return memory_mb
    except ImportError:
        return 0.0

def cleanup_memory():
    """Force garbage collection and memory cleanup."""
    import gc
    gc.collect()
    print("🧹 Memory cleanup completed")

def check_system_requirements():
    """Check system requirements and provide setup guidance."""
    print("🔍 System Requirements Check")
    print("=" * 40)
    
    # Check Python packages
    required_packages = {
        'yt_dlp': 'pip install yt-dlp',
        'pydub': 'pip install pydub', 
        'pandas': 'pip install pandas',
        'google.generativeai': 'pip install google-generativeai',
        'dotenv': 'pip install python-dotenv'
    }
    
    missing_packages = []
    for package, install_cmd in required_packages.items():
        try:
            __import__(package)
            print(f"✅ {package.replace('.', '-').replace('_', '-')}")
        except ImportError:
            print(f"❌ {package.replace('.', '-').replace('_', '-')} - Install: {install_cmd}")
            missing_packages.append(install_cmd)
    
    # Check ffmpeg
    ffmpeg_found = shutil.which('ffmpeg') is not None
    print(f"{'✅' if ffmpeg_found else '❌'} ffmpeg")
    
    if not ffmpeg_found:
        print("   Install: conda install -c conda-forge ffmpeg")
    
    # Check API key only if dotenv is available
    api_key = None
    try:
        if DOTENV_AVAILABLE:
            from dotenv import load_dotenv
            # Try to load .env files
            possible_env_paths = [
                ".env",
                "../.env", 
                "/teamspace/studios/this_studio/.env",
                "/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/.env"
            ]
            
            for env_path in possible_env_paths:
                if os.path.exists(env_path):
                    load_dotenv(dotenv_path=env_path)
                    break
        
        api_key = os.getenv("GEMINI_API_KEY")
    except:
        pass
    
    print(f"{'✅' if api_key else '❌'} GEMINI_API_KEY")
    
    if not api_key:
        print("   Set: Get key from https://aistudio.google.com")
        print("   Create .env file with: GEMINI_API_KEY=\"your_key_here\"")
    
    print("\n" + "=" * 40)
    
    if missing_packages:
        print("📦 Install missing packages:")
        for cmd in missing_packages:
            print(f"   {cmd}")
        print("\n💡 Quick install command:")
        print("   pip install google-generativeai python-dotenv")
        return False
    
    return ffmpeg_found and api_key

def install_missing_packages():
    """Helper function to install missing packages."""
    print("🔧 Installing missing packages...")
    print("Run these commands in your terminal:")
    print()
    print("pip install google-generativeai python-dotenv")
    print()
    print("Or install them one by one:")
    print("pip install google-generativeai")  
    print("pip install python-dotenv")
    print()
    print("After installation, restart your Python kernel and run the script again.")
    return False

# Example usage and demonstrations
if __name__ == "__main__":
    print("🎵 Enhanced SoundCloud Transcriber with Structured Data Output")
    print("=" * 70)
    print()
    
    # Check system requirements
    requirements_ok = check_system_requirements()
    
    if not requirements_ok:
        print("\n⚠️  Missing requirements detected!")
        print("📋 To install missing packages, run in your terminal:")
        print("   pip install google-generativeai python-dotenv")
        print()
        print("🔄 After installation, restart your Python kernel and run again.")
        # Don't exit, just show the usage examples
    
    print("\n📋 Usage Examples:")
    print("-" * 20)
    
    # Example 1: Single URL with structured output
    print("1️⃣  Single URL transcription:")
    print('   success, path, meta = stream_transcribe_single_url(')
    print('       "https://soundcloud.com/radio-ergo/idaacadda-01-jul-2024",')
    print('       output_format="structured"')
    print('   )')
    
    # Example 2: Date range with structured output
    print("\n2️⃣  Date range transcription:")
    print('   results = stream_transcribe_date_range(')
    print('       profile_url="https://soundcloud.com/radio-ergo",')
    print('       start_date="2024-07-01",')
    print('       end_date="2024-07-03",')
    print('       output_format="structured"')
    print('   )')
    
    # Example 3: View database statistics
    print("\n3️⃣  View statistics:")
    print('   stats = get_transcription_database_stats()')
    print('   print(f"Total: {stats[\'total_transcriptions\']}")')
    
    # Example 4: View recent transcriptions
    print("\n4️⃣  View recent transcriptions:")
    print('   recent = view_recent_transcriptions(n=5)')
    print('   print(recent)')
    
    print(f"\n📁 Default output directory:")
    print(f"   /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw")
    print(f"\n🗃️  Database file:")
    print(f"   transcriptions_database.csv")
    
    if requirements_ok:
        print("\n🚀 Ready to transcribe!")
    else:
        print("\n📦 Install missing packages first, then you'll be ready to transcribe!")

🎵 Enhanced SoundCloud Transcriber with Structured Data Output

🔍 System Requirements Check
✅ yt-dlp
✅ pydub
✅ pandas
✅ google-generativeai
✅ dotenv
✅ ffmpeg
✅ GEMINI_API_KEY


📋 Usage Examples:
--------------------
1️⃣  Single URL transcription:
   success, path, meta = stream_transcribe_single_url(
       "https://soundcloud.com/radio-ergo/idaacadda-01-jul-2024",
       output_format="structured"
   )

2️⃣  Date range transcription:
   results = stream_transcribe_date_range(
       profile_url="https://soundcloud.com/radio-ergo",
       start_date="2024-07-01",
       end_date="2024-07-03",
       output_format="structured"
   )

3️⃣  View statistics:
   stats = get_transcription_database_stats()
   print(f"Total: {stats['total_transcriptions']}")

4️⃣  View recent transcriptions:
   recent = view_recent_transcriptions(n=5)
   print(recent)

📁 Default output directory:
   /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw

🗃️  Database file:
   transcri

In [5]:
results = stream_transcribe_date_range(
    profile_url="https://soundcloud.com/radio-ergo",
    start_date="2024-07-01",
    end_date="2024-07-01",
    output_format="structured"
)

🔧 Checking for ffmpeg installation...
✓ Found working ffmpeg at: /usr/bin/ffmpeg
✓ Loaded API key from: ../.env
✓ Created structured database: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/transcriptions_database.csv

🎵 Streaming transcription: 2024-07-01 to 2024-07-01
📁 Output directory: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw
🧠 Using gemini for transcription
📊 Output format: structured
🔧 Using ffmpeg: /usr/bin/ffmpeg

[1/1] 2024-07-01
🔄 Processing: https://soundcloud.com/radio-ergo/idaacadda-01-jul-2024
⬇ Downloading to memory: https://soundcloud.com/radio-ergo/idaacadda-01-jul-2024
  🔧 Optimizing audio for transcription...                                
  📊 Processing 109.6MB audio in memory
  ⚠ Large audio file, processing may take longer
  🎯 Generating transcript...


/tmp/ipykernel_3848/883228011.py:643: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_row], ignore_index=True)


  ✓ Added to structured database: IDAACADDA-01-JUL-2024

🎵 STREAMING TRANSCRIPTION SUMMARY
  📊 Total URLs processed: 1
  ✅ Successfully transcribed: 1
  ⏭️  Skipped (already done): 0
  ❌ Failed: 0
  📁 Output directory: /teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw
  📊 Output format: structured
  🗃️  Structured database: transcriptions_database.csv
  🎧 Total audio duration: 59.8 minutes
  📝 Total words transcribed: 4,256
  ⏱️  Total processing time: 1.4 minutes
  💾 Memory-efficient processing: ✓
  🔧 ffmpeg status: ✓



In [7]:
import pandas as pd

def load_transcription_data(file_path: str) -> pd.DataFrame | None:
    """
    Loads transcription data from a CSV file into a pandas DataFrame.

    Args:
        file_path: The path to the CSV file.

    Returns:
        A pandas DataFrame containing the data, or None if the file is not found.
    """
    try:
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        print("File loaded successfully!")
        return df
    except FileNotFoundError:
        print(f"Error: The file was not found at the path: {file_path}")
        print("Please make sure the file is uploaded and the path is correct.")
        return None

# --- Usage Example ---

# Define the name of your file
# This assumes the file is in the same directory as your script
file_name = '/teamspace/studios/this_studio/somali-radios-with-ai-for-food-security/data/01_raw/transcriptions_database.csv'

# Load the data using the function
transcriptions_df = load_transcription_data(file_name)

# If the DataFrame was loaded successfully, display the first 5 rows
if transcriptions_df is not None:
    print("\nHere's a preview of your data:")
    print(transcriptions_df.head())

File loaded successfully!

Here's a preview of your data:
                                 id  \
0  d2fb77e8a8ab60bc5efc69d0cc721324   

                                                 url                  title  \
0  https://soundcloud.com/radio-ergo/idaacadda-01...  IDAACADDA 01-JUL-2024   

  source_type  date_recorded              date_processed transcription_method  \
0  soundcloud       20240701  2025-09-18T08:03:29.910372               gemini   

         model_used  processing_duration_seconds  audio_size_mb  ...  \
0  gemini-2.0-flash                     82.70929      82.173383  ...   

   audio_format audio_sample_rate  transcript_length_chars  \
0           mp3             48000                    23879   

   transcript_length_words  language_detected confidence_score  \
0                     4256             somali              NaN   

   processing_status error_message  file_path  \
0            success           NaN        NaN   

                                     tr

In [8]:
transcriptions_df

,id,url,title,source_type,date_recorded,date_processed,transcription_method,model_used,processing_duration_seconds,audio_size_mb,...,audio_format,audio_sample_rate,transcript_length_chars,transcript_length_words,language_detected,confidence_score,processing_status,error_message,file_path,transcript_text
0,d2fb77e8a8ab60bc5efc69d0cc721324,https://soundcloud.com/radio-ergo/idaacadda-01...,IDAACADDA 01-JUL-2024,soundcloud,20240701,2025-09-18T08:03:29.910372,gemini,gemini-2.0-flash,82.70929,82.173383,...,mp3,48000,23879,4256,somali,NaN,success,NaN,NaN,Halkan waxaad ka raadiyaha Ergo ee codka arima...
